# ARENA3 런타임 A — 심판

세 학습기를 같은 규칙으로 재는 **심판**입니다(진행 기록 208절, `docs/ARENA3_실행안내.md`). 강화학습은 여기서 하지 않습니다.

1. 런타임을 **CPU**로 고르고 **런타임 → 모두 실행**을 누르세요. Drive 접근을 허용하세요.
2. 순서: 차량 파일 대조 → **재현 점검**(원래 차량에서 P0·M4·W4가 이전 FIXED24 실행과 소수점까지 같아야 함, 다르면 멈춤) →
   기준선 252회(약 1시간) → **B′ 관문 평가** 84회(약 16시간, 둘로 나눠 동시에) → 판정.
3. 끊기면 다시 **모두 실행**을 누르세요. 끝난 실행은 해시를 확인한 뒤 건너뜁니다.
4. 끝나면 마지막 표를 저에게 보내 주세요. 모든 결과는 반복 사용 날짜의 개발 비교입니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_ARENA3.zip"
A3 = DRIVE / "arena3"
WORKSPACE = Path("/content/FutureTraffic_ARENA3")
assert ARCHIVE.is_file(), f"묶음이 없습니다: {ARCHIVE}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 결과 폴더:", A3)

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "ARENA3_V1", "ARENA3 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("arena3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(재현 점검이 실패할 수 있어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_arena3.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 평가 84상황의 차량 파일이 잠금과 같은지 확인합니다(SUMO 없이 1~3분).
colab.run_live(["src/arena3_common.py", "check-routes", "--work", "/content/a3_routes", "--part", "eval"], WORKSPACE)

In [ ]:
# 2) 재현 점검: 원래 배치(08-12 07:30, 난수 42)에서 P0·M4·W4가 FIXED24와 같아야 합니다. 다르면 여기서 멈춥니다.
BASE = A3 / "referee" / "baseline"
colab.run_live(["src/arena3_referee.py", "check", "--out", str(BASE), "--work", "/content/a3_check"], WORKSPACE)

In [ ]:
# 3) 기준선: 84상황 × P0·M4·W4 = 252회(약 1시간).
colab.run_live(["src/arena3_referee.py", "baseline", "--out", str(BASE), "--work", "/content/a3_base",
                "--workers", str(CPUS)], WORKSPACE)
display(Markdown((BASE / "summary.md").read_text()))

In [ ]:
# 4) B′ 관문 평가: 84상황을 둘로 나눠 동시에 돌립니다(각 롤아웃 작업자 CPU/2). 셀을 멈춰도 작업은 계속 돕니다.
#    5분마다 진행을 보여 줍니다. 약 16시간(선택 지점 수에 따라 10~24시간).
BEVAL = A3 / "referee" / "bprime"
LOGS = A3 / "logs" / "A"
half = max(1, CPUS // 2)
jobs = {f"bprime_eval_{i}": ["src/arena3_bprime.py", "run", "--role", "eval", "--out", str(BEVAL),
                             "--work", f"/content/b3e_{i}", "--shard", str(i), "--shards", "2",
                             "--rollout-workers", str(half)] for i in range(2)}

def describe(name):
    s = colab.read_json(BEVAL / "status" / f"eval_shard{name[-1]}.json")
    return None if not s else (f"{s['done']}/{s['total']} 끝, 실패 {len(s['failed'])}, "
                               f"복원 불일치 {s['restore_mismatched']}/{s['restore_checked']}")

pids = colab.launch(jobs, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe)
assert code == 0, "B′ 관문 평가에서 오류가 났습니다. 위 로그를 저에게 보내 주세요."

In [ ]:
# 5) 판정(M4 대비). G1은 런타임 C의 평가가 끝나면 함께 판정됩니다(없으면 건너뜀).
colab.run_live(["src/arena3_referee.py", "judge", "--baseline", str(BASE), "--candidate", f"W4={BASE}",
                "--candidate", f"BPRIME={BEVAL}", "--candidate", f"G1={A3 / 'referee' / 'G1'}",
                "--out", str(A3 / "referee")], WORKSPACE)
display(Markdown((A3 / "referee" / "summary.md").read_text()))